# 網易雲音樂數據整合與分析 (NetEase Music Data Analysis)

此 Notebook 用於處理網易 (NetEase/NCM) 平台的結算報表。其主要功能包括：
1. **遞迴讀取** 指定資料夾內的所有網易 Excel 檔案。
2. **排除總計表**（如：总计表、汇总、Summary 等）。
3. **排除每張 Sheet 的最後一列合計資料**。
4. **提取 Sheet Name** 作為新欄位 `date`（取最後 7 位字元 YYYY-MM）。
5. **欄位自動對齊**：對齊 `Song`、`Artist`、`ISRC`、`UPC`、`Album` 欄位。
6. **動態計算點擊與營收**：
   * **Clicks**：自動加總所有包含 **「量」** 或 **「数量」** 的欄位（如播放量、下載量、使用量、銷售數量）。
   * **Revenue**：自動加總所有包含 **「费用」** 或 **「收益费用」** 的欄位。
7. **資料清洗與缺失 ISRC 填補**（若 ISRC 為空，則填入 `Song - Artist`）。
8. **產出統計報告與排行報表**。

In [1]:
# 啟用自動重新載入外部模組功能
%load_ext autoreload
%autoreload 2

import os
import glob
import pandas as pd
import numpy as np

In [2]:
# 1. 設定輸入路徑與輸出路徑
base_path = '/Users/chu-chun/Mirror/Eva/input/sony_網易_valid/'
output_report_path = '../output/Netease_Song_Report.xlsx'
output_missing_isrc_path = '../output/Netease_Missing_ISRC_Report.xlsx'

In [3]:
# 2. 定義網易欄位對齊標準名稱的別名對照表 (不分大小寫)
netease_schema = {
    'Song': ['歌曲名', 'song', 'mv名', 'title', '歌曲名稱'],
    'Album': ['专辑', 'album', '專輯'],
    'Artist': ['艺人', 'artist', '歌手名', '歌手', '藝人'],
    'ISRC': ['isrc', '歌曲isrc', '歌曲 ISRC'],
    'UPC': ['upc', '专辑upc', '專輯 UPC']
}

# 欄位自動命名對應函數
def map_netease_columns(columns, schema):
    rename_map = {}
    for col in columns:
        col_str = str(col).strip().lower()
        for standard, aliases in schema.items():
            if col_str in [a.lower().strip() for a in aliases]:
                rename_map[col] = standard
                break
    return rename_map

# 判斷是否為總計表 (Summary Sheet) 的函數
def is_summary_sheet(sheet_name):
    ignored_keywords = ['总计', '總計', '汇总', 'summary', 'total']
    name_lower = sheet_name.lower().strip()
    return any(kw in name_lower for kw in ignored_keywords)

In [4]:
# 3. 讀取所有檔案的所有有效 Sheet，並記錄 Sheet Name 作為 date 欄位，同時動態加總點擊與收益
all_dfs = []
xlsx_files = glob.glob(os.path.join(base_path, '**/*.xlsx'), recursive=True)
# 自動排除 Excel 暫存鎖定檔案 (以 ~$ 開頭者)
xlsx_files = [f for f in xlsx_files if not os.path.basename(f).startswith('~$')]

print(f'🔍 開始掃描資料夾... 找到 {len(xlsx_files)} 個 Excel 檔案進行讀取。')

for file in xlsx_files:
    filename = os.path.basename(file)
    try: 
        xl = pd.ExcelFile(file)
        
        for sheet_name in xl.sheet_names:
            # 條件 1：不要讀總計表
            if is_summary_sheet(sheet_name):
                print(f'   [跳過] 總計表: [{sheet_name}] (檔案: {filename})')
                continue
                
            # 讀取該張 Sheet
            df_sheet = pd.read_excel(xl, sheet_name=sheet_name)
            
            # 🚨 額外條件：跳過最後一列資料 (因為是合計資料)
            if not df_sheet.empty:
                df_sheet = df_sheet.iloc[:-1]
            
            if df_sheet.empty:
                continue
                
            # 條件 2：將 sheet name 最後面 7 位字元 (例如 '2023-01') 當作新欄位 'date'
            df_sheet['date'] = sheet_name[-7:-3]
            
            # 計算 Clicks：動態加總所有欄位名稱含有「量」或「数量」的欄位值
            # (包括：总播放量、总下载量、销售数量、词使用量、曲使用量、原曲使用量、原伴奏使用量、消音伴奏使用量)
            # click_cols = [c for c in df_sheet.columns if '量' in str(c) or '数量' in str(c)]
            # if click_cols:
            #     df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
            #     print(f'      已加總點擊欄位: {click_cols}')
            # else:
            #     df_sheet['Clicks'] = 0.0

            # 按照表名稱計算 Clicks
            if sheet_name.startswith('免费'):
                click_cols = ['总播放量', '总下载量']
                df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
                print(f'      已加總點擊欄位: {click_cols}')
            elif sheet_name.startswith('付费单曲'):
                click_cols = ['销售数量']
                df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
                print(f'      已加總點擊欄位: {click_cols}')
            elif sheet_name.startswith('付费'):
                click_cols = ['总播放量', '总下载量']
                df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
                print(f'      已加總點擊欄位: {click_cols}')
            elif sheet_name.startswith('K歌'):
                click_cols = ['原曲使用量']
                df_sheet['Clicks'] = df_sheet[click_cols].fillna(0).sum(axis=1)
                print(f'      已加總點擊欄位: {click_cols}')
            else:
                df_sheet['Clicks'] = 0.0   
                
            # 計算 Revenue：動態加總所有欄位名稱含有「费用」或「收益费用」的欄位值
            # (包括：本月分成收益费用、本月实际分成收益费用、本月实际销售收益费用、本月单价收益费用)
            revenue_cols = [c for c in df_sheet.columns if '费用' in str(c) or '收益费用' in str(c)]
            if revenue_cols:
                df_sheet['Revenue'] = df_sheet[revenue_cols].fillna(0).sum(axis=1)
                print(f'      已加總營收欄位: {revenue_cols}')
            else:
                df_sheet['Revenue'] = 0.0
            
            # 條件 3：欄位標準化對齊
            rename_map = map_netease_columns(df_sheet.columns, netease_schema)
            df_sheet = df_sheet.rename(columns=rename_map)
            
            # 防呆：確保核心分析欄位至少都存在 (若該 Sheet 缺漏則補為 NaN)
            for target_col in ['Song', 'Artist', 'ISRC', 'Revenue', 'Clicks']:
                if target_col not in df_sheet.columns:
                    df_sheet[target_col] = np.nan
            
            # 記錄原始檔案名稱方便追蹤
            df_sheet['source_file'] = sheet_name +'/' +filename
            
            # 保留必要的欄位合併，避免欄位過多雜亂
            keep_cols = ['Song', 'Album', 'Artist', 'ISRC', 'UPC', 'Clicks', 'Revenue', 'date', 'source_file']
            df_filtered_cols = df_sheet[[c for c in keep_cols if c in df_sheet.columns]].copy()
            
            all_dfs.append(df_filtered_cols)
            print(f'   [載入] Sheet: [{sheet_name}] (共 {len(df_filtered_cols)} 筆) 來自: {filename}')
            
    except Exception as e:
        print(f'❌ 讀取檔案失敗: {filename}, 錯誤: {e}')

# 4. 合併所有 Sheet (不同 Sheet 欄位不同會自動對齊，缺失部分填補為 NaN)
if all_dfs:
    df_raw = pd.concat(all_dfs, ignore_index=True)
    print(
        f'\n' + '='*50 + f'\n' 
        f'🎉 合併完成！共讀取 {len(xlsx_files)} 個檔案，累計 {len(df_raw)} 筆原始列資料。\n' 
        f'' + '='*50
    )
else:
    df_raw = pd.DataFrame()
    print('❌ 警告：未成功讀取到任何資料，請確認資料夾與檔案名稱。')

🔍 開始掃描資料夾... 找到 8 個 Excel 檔案進行讀取。
   [跳過] 總計表: [总计表] (檔案: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx)
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-10] (共 355 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-11] (共 356 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月分成收益费用']
   [載入] Sheet: [免费2023-12] (共 361 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-10] (共 359 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-11] (共 358 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2023.10-2023.12).xlsx
      已加總點擊欄位: ['总播放量', '总下载量']
      已加總營收欄位: ['本月实际分成收益费用']
   [載入] Sheet: [付费2023-12] (共 360 筆) 來自: 罗大佑曲库（2021.01-2024.01）-结算周期(2

In [5]:
# 4. 資料清理與 ISRC 填補
df_cleaned = df_raw.copy()

if not df_cleaned.empty:
    # 4-1. 將歌名、歌手強制轉為乾淨的字串型態，並防呆小數點
    for col in ['Song', 'Artist']:
        if col in df_cleaned.columns:
            df_cleaned[col] = df_cleaned[col].apply(
                lambda x: str(int(x)) if isinstance(x, float) and x.is_integer()
                          else (str(x).strip() if pd.notna(x) else x)
            )
            
    # 4-2. 如果 ISRC 為空值，自動填入 "Song - Artist" 的組合
    if 'ISRC' in df_cleaned.columns and 'Song' in df_cleaned.columns and 'Artist' in df_cleaned.columns:
        isrc_fill = df_cleaned['Song'].fillna('UnknownSong').astype(str) + ' - ' + df_cleaned['Artist'].fillna('UnknownArtist').astype(str)
        df_cleaned['ISRC'] = df_cleaned['ISRC'].fillna(isrc_fill)
        df_cleaned.loc[df_cleaned['ISRC'].astype(str).str.strip() == '', 'ISRC'] = isrc_fill
        
    # 4-3. 統一將 ISRC 轉為大寫並去除空白
    if 'ISRC' in df_cleaned.columns:
        df_cleaned['ISRC'] = df_cleaned['ISRC'].astype(str).str.upper().str.strip()
        
    print('✅ 資料清洗與 ISRC 填補完成！')
    df_cleaned.info()
else:
    print('無資料可供清理。')

✅ 資料清洗與 ISRC 填補完成！
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11737 entries, 0 to 11736
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Song         11737 non-null  object 
 1   Album        11737 non-null  object 
 2   Artist       11737 non-null  object 
 3   ISRC         11737 non-null  object 
 4   UPC          58 non-null     float64
 5   Clicks       11737 non-null  int64  
 6   Revenue      11737 non-null  float64
 7   date         11737 non-null  object 
 8   source_file  11737 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 825.4+ KB


In [6]:
# # 5. 產出「缺失 ISRC 歌曲」統計報表 (一列算一筆)
# if not df_cleaned.empty:
#     # 判斷是否為當初缺 ISRC 的列 (欄位等於當時填補的 'Song - Artist' 格式)
#     temp_fill = df_cleaned['Song'].fillna('UnknownSong').astype(str) + ' - ' + df_cleaned['Artist'].fillna('UnknownArtist').astype(str)
#     isrc_missing = df_cleaned['ISRC'] == temp_fill
#     df_missing = df_cleaned[isrc_missing]
    
#     print(f'📊 缺失 ISRC 原始資料共計 {len(df_missing)} 行。')
    
#     if not df_missing.empty:
#         # 以 歌曲 + 歌手 分組，加總營收、統計筆數 (一列一筆)、列出來源檔案
#         missing_report = (
#             df_missing
#             .groupby(['Song', 'Artist'])
#             .agg(
#                 revenue=('Revenue', 'sum'),
#                 records=('Song', 'size'),  # 統計筆數 (一列算一筆)
#                 source_files=('source_file', lambda x: ', '.join(sorted(list(set(x.dropna())))))
#             )
#             .reset_index()
#         )
        
#         # 依照筆數由高到低進行排序
#         missing_report = missing_report.sort_values(by='records', ascending=False).reset_index(drop=True)
        
#         # 匯出至 Excel
#         os.makedirs(os.path.dirname(output_missing_isrc_path), exist_ok=True)
#         missing_report.to_excel(output_missing_isrc_path, index=False)
        
#         print(f'💾 缺失 ISRC 報表已匯出至：{output_missing_isrc_path}')
#         display(missing_report.head(15))
#     else:
#         print('🎉 太棒了！沒有任何缺失 ISRC 的歌曲。')
# else:
#     print('無資料可處理。')

## align artist name

In [18]:
# 1. 如今才是唯一	['罗大佑,娃娃' '娃娃,罗大佑' '罗大佑']
condition1 = (df_cleaned['Song'] == '如今才是唯一') & (df_cleaned['Artist'] == '娃娃,罗大佑')
df_cleaned.loc[condition1, 'ISRC'] = '如今才是唯一 - 罗大佑,娃娃'

# 2. 滚滚红尘	['罗大佑,陈淑桦' '袁凤瑛' '陈淑桦,罗大佑']
condition2 = (df_cleaned['Song'] == '滚滚红尘') & (df_cleaned['Artist'] == '陈淑桦,罗大佑')
df_cleaned.loc[condition2, 'ISRC'] = '滚滚红尘 - 罗大佑,陈淑桦'



In [19]:
# 6. 產出「網易歌曲營收總排行報表」
if not df_cleaned.empty:
    song_report = (
        df_cleaned
        .groupby(['ISRC', 'date'])
        .agg(
            song=('Song', 'first'),
            artist=('Artist', 'first'),
            revenue=('Revenue', 'sum'),
            clicks=('Clicks', 'sum')
        )
        .reset_index()
    )
    
    # 依照總收益由高到低排序
    song_report = song_report.sort_values(by=['date', 'revenue'], ascending=False).reset_index(drop=True)
    
    # 匯出至 Excel
    # os.makedirs(os.path.dirname(output_report_path), exist_ok=True)
    # song_report.to_excel(output_report_path, index=False)
    
    print(f'💾 網易歌曲營收總排行報表已匯出至：{output_report_path}')
    display(song_report.head(15))
else:
    print('無資料可處理。')

💾 網易歌曲營收總排行報表已匯出至：../output/Netease_Song_Report.xlsx


,ISRC,date,song,artist,revenue,clicks
0,似是故人来 - 梅艳芳,2024,似是故人来,梅艳芳,702.308050,366291
1,天若有情 - 袁凤瑛,2024,天若有情,袁凤瑛,47.436025,32692
2,青春舞曲2000 - 群星,2024,青春舞曲2000,群星,24.342580,10736
3,新生代 - 群星,2024,新生代,群星,11.380197,5207
4,东方之珠 - 罗大佑,2024,东方之珠,罗大佑,8.497184,4127
5,道 - 黄霑,2024,道,黄霑,4.792872,2650
6,情人眼里 - 袁凤瑛,2024,情人眼里,袁凤瑛,2.973103,1669
7,"只要是爱 - 罗大佑,袁凤瑛",2024,只要是爱,"罗大佑,袁凤瑛",2.303489,1135
8,童年 (INSTRUMENTAL)(LIVE) - 罗大佑,2024,童年 (Instrumental)(Live),罗大佑,1.911358,846
9,出走 - 夏韶声,2024,出走,夏韶声,0.979509,501


In [20]:
# transform the df to pivot table
song_report

,ISRC,date,song,artist,revenue,clicks
0,似是故人来 - 梅艳芳,2024,似是故人来,梅艳芳,702.308050,366291
1,天若有情 - 袁凤瑛,2024,天若有情,袁凤瑛,47.436025,32692
2,青春舞曲2000 - 群星,2024,青春舞曲2000,群星,24.342580,10736
3,新生代 - 群星,2024,新生代,群星,11.380197,5207
4,东方之珠 - 罗大佑,2024,东方之珠,罗大佑,8.497184,4127
...,...,...,...,...,...,...
364,BASTARD - YSL NUCLEAR,2023,bastard,YSL Nuclear,0.190223,47
365,小丑 - FOCUS乐团,2023,小丑,FOCUS乐团,0.035155,18
366,孤巴察峨 GUBATSAEH - FOCUS乐团,2023,孤巴察峨 Gubatsaeh,FOCUS乐团,0.030833,23
367,BOXING - FOCUS乐团,2023,Boxing,FOCUS乐团,0.008199,11


In [21]:
song_pivot = song_report.pivot_table(
    index=['ISRC', 'song', 'artist'],
    columns='date',
    values=['clicks', 'revenue'],
    aggfunc='sum',
    fill_value=0
)

# 4. Calculate horizontal sums using MultiIndex slicing
click_cols = [col for col in song_pivot.columns if col[0] == 'clicks']
rev_cols = [col for col in song_pivot.columns if col[0] == 'revenue']

song_pivot[('total_click', '')] = song_pivot[click_cols].sum(axis=1)
song_pivot[('total_revenue', '')] = song_pivot[rev_cols].sum(axis=1)

# Extract year list
years = sorted(list(set(y for metric, y in song_pivot.columns if y != '' and metric in ['clicks', 'revenue'])))

# Reset index to move row index into columns
song_pivot_df = song_pivot.reset_index()

# 5. Flatten columns and reorder
flat_cols = []
for col in song_pivot_df.columns:
    level0, level1 = col
    if level1 == '':
        flat_cols.append(level0)
    else:
        flat_cols.append(f"{level1}_{level0}")
        
song_pivot_df.columns = flat_cols

# Build desired column order (only including ISRC, song, artist, total metrics, and year metrics)
base_cols = ['ISRC', 'song', 'artist']
summary_cols = ['total_click', 'total_revenue']

# Group all years' clicks first, then all years' revenue
time_cols = [f"{y}_clicks" for y in years] + [f"{y}_revenue" for y in years]
    
final_cols = base_cols + summary_cols + time_cols
song_final = song_pivot_df[final_cols].copy()

# Sort by total revenue descending
song_final = song_final.sort_values('total_revenue', ascending=False).reset_index(drop=True)

# Export to Excel
song_final.to_excel(output_report_path, index=False)

In [22]:
temp = song_final.groupby('song')['artist'].unique()
result = temp[temp.apply(len) > 1]
df_result = pd.DataFrame(result).reset_index()
df_result.to_excel("/Users/chu-chun/Mirror/Eva/output/NetEase_multiple_singer_list.xlsx", index=False)

